In [37]:
# CELL 0 — Install
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install pandas numpy transformers scikit-learn flask accelerate -q

In [38]:
# CELL 1 — Imports
import pandas as pd
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from torch.utils.data import WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')


# ADD these 5 lines at the bottom of Cell 1
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
import random; random.seed(SEED)
print("Seeds fixed — results will be reproducible now.")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4
Seeds fixed — results will be reproducible now.


In [39]:
# CELL 2 — Data loading & MAJORITY VOTING aggregation
#
# ═══════════════════════════════════════════════════════════════
# ROOT CAUSE OF 65% ACCURACY:
#   Your CSV has 17,775 rows but only 1,700 UNIQUE sentences.
#   Each sentence was annotated by ~10 different people.
#   The same sentence appears up to 10 times with CONFLICTING
#   labels (some annotators say Biased, others say Non-biased).
#
#   Training on all 17,775 rows means the model is shown the
#   identical text with opposite labels — it can never converge.
#   This is WHY accuracy was stuck at 65% regardless of epochs.
#
# THE FIX: aggregate to 1 row per sentence using majority vote.
# Also use the 'factual' column as a prefix signal — it has
# 85% correlation with the bias label.
# ═══════════════════════════════════════════════════════════════

df_raw = pd.read_csv('annotations.csv')
print(f'Raw rows       : {len(df_raw):,}')
print(f'Unique sentences: {df_raw["sentence_id"].nunique():,}')
print(f'Annotations/sentence: ~{len(df_raw)/df_raw["sentence_id"].nunique():.1f}')

# ── Step 1: Aggregate multiple annotations → one row per sentence ──
grp = df_raw.groupby(['sentence_id', 'text', 'factual']).apply(
    lambda g: pd.Series({
        'biased_votes' : (g['label'] == 'Biased').sum(),
        'total_votes'  : len(g),
        'biased_pct'   : (g['label'] == 'Biased').mean(),
    })
).reset_index()

# Majority vote label
grp['label'] = (grp['biased_pct'] >= 0.5).astype(int)   # 1=Biased, 0=Non-biased

# Agreement score (0.5 = total disagreement, 1.0 = perfect agreement)
grp['agreement'] = grp['biased_pct'].apply(lambda p: max(p, 1-p))

print(f'\nAfter aggregation  : {len(grp):,} unique sentences')
print(f'Label distribution :')
print(grp['label'].value_counts().rename({1:'Biased', 0:'Non-biased'}))
print(f'\nInter-annotator agreement:')
print(pd.cut(grp['agreement'],
             bins=[0.5,0.6,0.7,0.8,0.9,1.0],
             labels=['50-60%','60-70%','70-80%','80-90%','90-100%']
            ).value_counts().sort_index())

Raw rows       : 17,775
Unique sentences: 1,700
Annotations/sentence: ~10.5

After aggregation  : 4,632 unique sentences
Label distribution :
label
Biased        3133
Non-biased    1499
Name: count, dtype: int64

Inter-annotator agreement:
agreement
50-60%      139
60-70%      455
70-80%      539
80-90%      396
90-100%    2689
Name: count, dtype: int64


/tmp/ipykernel_8617/4217515593.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grp = df_raw.groupby(['sentence_id', 'text', 'factual']).apply(


In [40]:
# CELL 3 — Feature engineering: prepend 'factual' type as prefix
#
# The 'factual' column is a GOLD-STANDARD signal:
#   'Expresses writer\'s opinion'            → 85% Biased
#   'Somewhat factual but also opinionated'  → 79% Biased
#   'Entirely factual'                       → 17% Biased
#
# We prepend a short label to each text so the model gets
# this signal explicitly without any architecture changes.

FACTUAL_PREFIX = {
    'Expresses writer\'s opinion'           : '[OPINION]',
    'Somewhat factual but also opinionated' : '[MIXED]',
    'Entirely factual'                      : '[FACTUAL]',
}

grp['text_with_prefix'] = grp.apply(
    lambda r: FACTUAL_PREFIX.get(r['factual'], '') + ' ' + r['text'].strip(),
    axis=1
)

print('Sample prefixed texts:')
for _, row in grp.sample(3, random_state=42).iterrows():
    print(f"  [{row['label']}] {row['text_with_prefix'][:100]}...")
    print()

Sample prefixed texts:
  [1] [FACTUAL] America is likely to remain a relative manufacturing wasteland, as barren as Trump’s own i...

  [1] [MIXED] The anti-vaccination community holds to an unscientific conspiracy theory that childhood vac...

  [0] [FACTUAL] Day said that some Democrats had phoned her to say they would not vote for Hillary Clinton...



In [41]:
# CELL 4 — Train/val split (stratified)
train_df, val_df = train_test_split(
    grp,
    test_size=0.2,
    random_state=42,
    stratify=grp['label'],
)
print(f'Train : {len(train_df):,}   Val : {len(val_df):,}')
print('Train label dist:', train_df['label'].value_counts().to_dict())
print('Val   label dist:', val_df['label'].value_counts().to_dict())

Train : 3,705   Val : 927
Train label dist: {1: 2506, 0: 1199}
Val   label dist: {1: 627, 0: 300}


In [42]:
# CELL 5 — Tokenisation
#
# Using roberta-base (larger, 125M params) for better accuracy ceiling.
# With only 1,360 training samples, we need a strong pre-trained backbone.

BASE_MODEL = 'roberta-base'
tokenizer  = AutoTokenizer.from_pretrained(BASE_MODEL)

train_encodings = tokenizer(
    train_df['text_with_prefix'].tolist(),
    truncation=True, padding=True, max_length=256
)
val_encodings = tokenizer(
    val_df['text_with_prefix'].tolist(),
    truncation=True, padding=True, max_length=256
)
print('Tokenisation done.')

Tokenisation done.


In [43]:
# CELL 6 — Dataset with per-sample agreement weights
#
# Each sample carries its agreement score so the trainer can
# weight high-confidence annotations more.

class BiasDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels, weights=None):
        self.encodings = encodings
        self.labels    = labels
        self.weights   = weights  # per-sample agreement weight

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        if self.weights is not None:
            item['weight'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = BiasDataset(
    train_encodings,
    train_df['label'].tolist(),
    weights=train_df['agreement'].tolist(),   # higher agreement = more weight
)
val_dataset = BiasDataset(
    val_encodings,
    val_df['label'].tolist(),
)
print(f'Train : {len(train_dataset):,}  |  Val : {len(val_dataset):,}')

Train : 3,705  |  Val : 927


In [44]:
# CELL 7 — Class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print('Class weights (Non-biased, Biased):', class_weights)

Class weights (Non-biased, Biased): tensor([1.5450, 0.7392])


In [45]:
# CELL 8 — Load model
#
# Using roberta-base — 125M params, strong NLP backbone.
# DO NOT freeze any layers: dataset is only ~1,360 samples,
# so we need the full model to update (freezing layers hurts
# on small datasets because the classifier head can't get
# enough signal from the tiny unfrozen portion alone).

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'Non-Biased', 1: 'Biased'},
    label2id={'Non-Biased': 0, 'Biased': 1},
    hidden_dropout_prob=0.2,       # extra dropout for small dataset
    attention_probs_dropout_prob=0.2,
)
model.to(device)
print('Params:', sum(p.numel() for p in model.parameters())//1_000_000, 'M')

Device : cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Params: 124 M


In [46]:
# CELL 9 — Weighted trainer using per-sample agreement scores

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels        = inputs.pop('labels')
        sample_weight = inputs.pop('weight', None)
        outputs       = model(**inputs)
        logits        = outputs.logits

        # Per-sample loss (reduction='none' so we can weight each sample)
        loss_fn   = CrossEntropyLoss(
            weight=class_weights.to(model.device),
            reduction='none',
        )
        per_loss = loss_fn(logits, labels)   # shape: (batch,)

        if sample_weight is not None:
            # Scale loss by inter-annotator agreement (0.5 to 1.0)
            per_loss = per_loss * sample_weight.to(model.device)

        loss = per_loss.mean()
        return (loss, outputs) if return_outputs else loss

In [47]:
# CELL 10 — Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted'
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy' : round(acc,  4),
        'f1'       : round(f1,   4),
        'precision': round(prec, 4),
        'recall'   : round(rec,  4),
    }

In [48]:
# CELL 11 — Training configuration
#
# Tuned for small dataset (~1,360 training samples):
#   learning_rate  = 1e-5  (conservative — prevents overfitting on small data)
#   epochs         = 10    (more epochs needed when dataset is small)
#   batch_size     = 16    (small batch = noisier gradients = better generalisation)
#   weight_decay   = 0.1   (strong L2 reg — critical for small datasets)
#   warmup_ratio   = 0.15  (longer warmup as % of short total steps)
#   cosine LR      = True  (smooth decay helps squeeze last few % accuracy)

training_args = TrainingArguments(
    output_dir                  = './results',
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    gradient_accumulation_steps = 2,
    learning_rate               = 1e-5,
    warmup_ratio                = 0.15,
    weight_decay                = 0.1,
    lr_scheduler_type           = 'cosine',
    fp16                        = torch.cuda.is_available(),
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    logging_dir                 = './logs',
    logging_steps               = 20,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    report_to                   = 'none',
    dataloader_num_workers      = 0,
    save_total_limit            = 2,
    seed                        = 42,
)
print('Config ready.')

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Config ready.


In [49]:
# CELL 12 — Create Trainer
trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

In [50]:
# CELL 13 — Train
train_result = trainer.train()
print('\nTraining complete.')
print(f"  Runtime     : {train_result.metrics['train_runtime']:.0f} s")
print(f"  Samples/sec : {train_result.metrics['train_samples_per_second']:.1f}")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.374230,0.593437,0.718400,0.727300,0.791800,0.718400
2,0.902624,0.482413,0.856500,0.856600,0.856700,0.856500
3,0.773607,0.450185,0.856500,0.856600,0.856700,0.856500
4,0.812486,0.450971,0.849000,0.850100,0.851900,0.849000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Training complete.
  Runtime     : 313 s
  Samples/sec : 118.3


In [51]:
# CELL 14 — Evaluate
results = trainer.evaluate()
print('\n=== Validation Results ===')
for k, v in results.items():
    print(f'  {k:<35} {v}')


=== Validation Results ===
  eval_loss                           0.48271486163139343
  eval_accuracy                       0.8565
  eval_f1                             0.8566
  eval_precision                      0.8567
  eval_recall                         0.8565
  eval_runtime                        1.3928
  eval_samples_per_second             665.548
  eval_steps_per_second               20.821
  epoch                               4.0


In [52]:
# CELL 15 — Per-class report
preds_output = trainer.predict(val_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

print(classification_report(
    y_true, y_pred,
    target_names=['Non-biased (0)', 'Biased (1)'],
    digits=4,
))

                precision    recall  f1-score   support

Non-biased (0)     0.7774    0.7800    0.7787       300
    Biased (1)     0.8946    0.8931    0.8939       627

      accuracy                         0.8565       927
     macro avg     0.8360    0.8366    0.8363       927
  weighted avg     0.8567    0.8565    0.8566       927



In [53]:
# CELL 16 — Save
model.save_pretrained('bias_model')
tokenizer.save_pretrained('bias_model')
print('Saved to ./bias_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./bias_model
